# Step 2: Heckman Control Function + DR-AIPW

Estima el efecto causal de **T** (homeownership binario) sobre **Y** (default).

Comparación:
- **Naive**: DR-AIPW solo sobre `accepted_labeled.csv` (sin BASL, sin Heckman)
- **BASL only**: logit sobre `D_tilde.csv` sin corrección de endogeneidad
- **DR-AIPW only**: Heckman + DR-AIPW sobre `accepted_labeled.csv` sin BASL
- **Ours**: BASL + Heckman + DR-AIPW sobre `D_tilde.csv`

ATE verdadero cargado de `data/tau_true.npy` (calculado por Monte Carlo en el DGP).

outline:

BASL genera D~.

Estimas una ecuación de tratamiento.

Construyes λ.

Añades λ a los nuisance models.

Estimas el efecto con DR-AIPW.

In [44]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.special import ndtr  # Phi(x)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm
from scipy.stats import norm

## 1. Cargar datos y construir tratamiento sintético

- **x1** (tratamiento): continuo

In [45]:
def load_and_prepare(path, treatment_col="T", target_col="y",
                     instrument_col="Z", seed=42):
    """
    Carga el dataset.
    - T: tratamiento binario (homeownership), ya en el CSV desde el DGP
    - Y: outcome (default)
    - X: covariables de control (X2, X3, X4)
    - Z: instrumento (acceso regional a subsidio habitacional)
    """
    df = pd.read_csv(path)
    drop_cols = [c for c in ["split", "Unnamed: 0"] if c in df.columns]
    df = df.drop(columns=drop_cols)

    # Dropear filas sin Z (por si D_tilde tiene filas sin instrumento)
    if instrument_col in df.columns:
        n_antes = len(df)
        df = df.dropna(subset=[instrument_col])
        n_dropped = n_antes - len(df)
        if n_dropped > 0:
            print(f"  Dropped {n_dropped} filas sin Z ({n_dropped/n_antes:.1%})")

    feature_cols = ["X2", "X3", "X4"]

    Y = df[target_col].values
    T = df[treatment_col].values.astype(int)
    X = df[feature_cols].values
    Z = df[instrument_col].values if instrument_col in df.columns else np.zeros(len(df))

    print(f"N = {len(df):,}")
    print(f"Bad rate (Y=1):       {Y.mean():.1%}")
    print(f"Treatment rate (T=1): {T.mean():.1%}")
    print(f"Corr(Z, T):           {np.corrcoef(Z, T)[0,1]:.3f}  (relevancia del instrumento)")
    print(f"Corr(Z, Y):           {np.corrcoef(Z, Y)[0,1]:.3f}  (exclusion restriction)")

    return Y, T, X, Z, feature_cols


# Cargar ATE verdadero del DGP (calculado por Monte Carlo en synthetic_data)
ATE_true = float(np.load("data/tau_true.npy"))
print(f"ATE verdadero (Monte Carlo): {ATE_true:.4f}\n")

print("=== D_tilde (BASL-expanded) ===")
Y_exp, T_exp, X_exp, Z_exp, feat_cols = load_and_prepare("data/D_tilde.csv")

print("\n=== Accepted only (baseline) ===")
Y_acc, T_acc, X_acc, Z_acc, _ = load_and_prepare("data/accepted_labeled.csv")

ATE verdadero (Monte Carlo): 0.0178

=== D_tilde (BASL-expanded) ===
N = 11,078
Bad rate (Y=1):       43.8%
Treatment rate (T=1): 46.8%
Corr(Z, T):           0.190  (relevancia del instrumento)
Corr(Z, Y):           0.243  (exclusion restriction)

=== Accepted only (baseline) ===
N = 14,111
Bad rate (Y=1):       35.5%
Treatment rate (T=1): 42.2%
Corr(Z, T):           0.157  (relevancia del instrumento)
Corr(Z, Y):           0.077  (exclusion restriction)


In [46]:
Z_exp

array([ 0.99402118, -0.32593945, -0.9254882 , ...,  1.12373958,
        0.97234364,  0.75382288], shape=(11078,))

## 2. Heckman Control Function

**First stage**: probit de T sobre (X, Z)  
**Control function**: inverse Mills ratio λ = φ(ν̂) / Φ(ν̂)  
λ captura la parte de la asignación de tratamiento explicada por factores latentes no observados.

In [47]:
import statsmodels.api as sm
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def heckman_control_function(T, X, Z, scaler=None):

    XZ = np.column_stack([X, Z.reshape(-1,1)])

    if scaler is None:
        scaler = StandardScaler()
        XZ_scaled = scaler.fit_transform(XZ)
    else:
        XZ_scaled = scaler.transform(XZ)

    XZ_const = sm.add_constant(XZ_scaled)

    probit = sm.Probit(T, XZ_const)
    res = probit.fit(disp=False)

    nu_hat = res.predict(XZ_const, linear=True)

    phi_nu = norm.pdf(nu_hat)
    Phi_nu = norm.cdf(nu_hat)

    lambda_hat = phi_nu / np.clip(Phi_nu, 1e-8, 1)

    T_pred = res.predict(XZ_const)

    auc_fs = roc_auc_score(T, T_pred)

    print(f"First stage AUC: {auc_fs:.4f}")
    print(f"lambda mean: {lambda_hat.mean():.4f}")
    print(f"lambda std:  {lambda_hat.std():.4f}")

    return lambda_hat, nu_hat, res, scaler

print("--- BASL-expanded ---")
lambda_exp, nu_exp, logit_exp, scaler_exp = heckman_control_function(T_exp, X_exp, Z_exp)

print("\n--- Accepted only ---")
lambda_acc, nu_acc, logit_acc, scaler_acc = heckman_control_function(T_acc, X_acc, Z_acc)

--- BASL-expanded ---
First stage AUC: 0.6149
lambda mean: 0.8584
lambda std:  0.1646

--- Accepted only ---
First stage AUC: 0.5903
lambda mean: 0.9343
lambda std:  0.1371


## 3. DR-AIPW con cross-fitting

Estima el PATE usando el espacio de covariables aumentado **(X, λ)**.

$$\hat{\tau}^{DR} = \frac{1}{n}\sum_i \left[ \frac{T_i(Y_i - \hat{\mu}_1)}{\hat{e}} - \frac{(1-T_i)(Y_i - \hat{\mu}_0)}{1-\hat{e}} + \hat{\mu}_1 - \hat{\mu}_0 \right]$$

Con **5-fold cross-fitting** para evitar overfitting en la estimación de los nuisance models.

In [48]:
def dr_aipw_crossfit(Y, T, X, lambda_hat, n_folds=5, seed=42):
    """
    DR-AIPW para tratamiento BINARIO con cross-fitting.
    Nuisance models:
    - e(X,lambda): propensity score → GradientBoostingClassifier
    - mu_t(X,lambda): outcome model → GradientBoostingClassifier
    """
    W = np.column_stack([X, lambda_hat.reshape(-1, 1)])
    n = len(Y)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    e_hat   = np.zeros(n)
    mu1_hat = np.zeros(n)
    mu0_hat = np.zeros(n)

    for fold, (train_idx, val_idx) in enumerate(kf.split(W)):
        W_tr, W_val = W[train_idx], W[val_idx]
        Y_tr = Y[train_idx]
        T_tr = T[train_idx]
        T_val = T[val_idx] 

        # Propensity score
        ps_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        ps_model.fit(W_tr, T_tr)
        e_hat[val_idx] = np.clip(ps_model.predict_proba(W_val)[:, 1], 0.01, 0.99)

        # Outcome model T=1
        idx1 = train_idx[T_tr == 1]
        om1  = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        om1.fit(W[idx1], Y[idx1])
        mu1_hat[val_idx] = om1.predict_proba(W_val)[:, 1]

        # Outcome model T=0
        idx0 = train_idx[T_tr == 0]
        om0  = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        om0.fit(W[idx0], Y[idx0])
        mu0_hat[val_idx] = om0.predict_proba(W_val)[:, 1]

        print(f"  Fold {fold+1}: PS AUC={roc_auc_score(T_val, e_hat[val_idx]):.3f}")

    # DR-AIPW scores
    scores = (
        T * (Y - mu1_hat) / e_hat
        - (1 - T) * (Y - mu0_hat) / (1 - e_hat)
        + mu1_hat - mu0_hat
    )
    tau_hat = scores.mean()

    # Bootstrap SE
    rng_b = np.random.default_rng(seed)
    boot_taus = []
    for _ in range(500):
        idx_b = rng_b.integers(0, n, size=n)
        boot_taus.append(scores[idx_b].mean())
    se = np.std(boot_taus)

    return tau_hat, se, scores


print("=" * 50)
print("DR-AIPW — BASL-expanded (D~)")
print("=" * 50)
tau_exp, se_exp, scores_exp = dr_aipw_crossfit(Y_exp, T_exp, X_exp, lambda_exp)

print("\n" + "=" * 50)
print("DR-AIPW — Accepted only (baseline)")
print("=" * 50)
tau_acc, se_acc, scores_acc = dr_aipw_crossfit(Y_acc, T_acc, X_acc, lambda_acc)

DR-AIPW — BASL-expanded (D~)
  Fold 1: PS AUC=0.600
  Fold 2: PS AUC=0.588
  Fold 3: PS AUC=0.616
  Fold 4: PS AUC=0.625
  Fold 5: PS AUC=0.610

DR-AIPW — Accepted only (baseline)
  Fold 1: PS AUC=0.575
  Fold 2: PS AUC=0.560
  Fold 3: PS AUC=0.566
  Fold 4: PS AUC=0.591
  Fold 5: PS AUC=0.579


## 4. Resultados y comparación

In [49]:
from scipy.stats import norm as normal

def report(label, tau, se, tau_true=None):
    z     = tau / se
    pval  = 2 * (1 - normal.cdf(abs(z)))
    ci_lo = tau - 1.96 * se
    ci_hi = tau + 1.96 * se
    sig   = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    bias_str = f"  Bias = {abs(tau - tau_true):.4f}" if tau_true is not None else ""
    print(f"{label}")
    print(f"  tau (PATE) = {tau:+.4f} {sig}")
    print(f"  SE         = {se:.4f}")
    print(f"  95% CI     = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    print(f"  p-value    = {pval:.4f}{bias_str}")
    print()

print("=" * 60)
print("EFECTO CAUSAL DE T (homeownership) SOBRE Y (default)")
print("Heckman Control Function + DR-AIPW")
print("=" * 60)
print(f"ATE verdadero: {ATE_true:.4f}\n")
report("1. Ours — BASL + Heckman + DR-AIPW", tau_exp, se_exp, ATE_true)
report("2. Naive — solo D^a sin corrección", tau_acc, se_acc, ATE_true)

diff = tau_exp - tau_acc
print("-" * 60)
print(f"Diferencia (BASL+Heckman - Naive): {diff:+.4f}")
print("=" * 60)

EFECTO CAUSAL DE T (homeownership) SOBRE Y (default)
Heckman Control Function + DR-AIPW
ATE verdadero: 0.0178

1. Ours — BASL + Heckman + DR-AIPW
  tau (PATE) = +0.1321 ***
  SE         = 0.0091
  95% CI     = [+0.1143, +0.1499]
  p-value    = 0.0000  Bias = 0.1143

2. Naive — solo D^a sin corrección
  tau (PATE) = +0.0503 ***
  SE         = 0.0086
  95% CI     = [+0.0334, +0.0672]
  p-value    = 0.0000  Bias = 0.0325

------------------------------------------------------------
Diferencia (BASL+Heckman - Naive): +0.0818


## Benchmarks


In [50]:
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np

def dr_aipw_binary_crossfit(Y, T, X, lambda_hat,
                            n_folds=5,
                            seed=42,
                            n_boot=500):

    W = np.column_stack([X, lambda_hat.reshape(-1, 1)])

    n = len(Y)

    e_hat   = np.zeros(n)
    mu1_hat = np.zeros(n)
    mu0_hat = np.zeros(n)

    kf = KFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=seed
    )

    for fold, (train_idx, val_idx) in enumerate(kf.split(W)):

        W_tr, W_val = W[train_idx], W[val_idx]
        Y_tr = Y[train_idx]
        T_tr = T[train_idx]

        # --------------------------------------------------
        # Propensity score e(X,λ)
        # --------------------------------------------------
        ps_model = GradientBoostingClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=seed
        )

        ps_model.fit(W_tr, T_tr)

        e_hat[val_idx] = ps_model.predict_proba(W_val)[:, 1]

        # --------------------------------------------------
        # μ1(X,λ)
        # --------------------------------------------------
        mu1_model = GradientBoostingClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=seed
        )

        mu1_model.fit(
            W_tr[T_tr == 1],
            Y_tr[T_tr == 1]
        )

        mu1_hat[val_idx] = mu1_model.predict_proba(W_val)[:, 1]

        # --------------------------------------------------
        # μ0(X,λ)
        # --------------------------------------------------
        mu0_model = GradientBoostingClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=seed
        )

        mu0_model.fit(
            W_tr[T_tr == 0],
            Y_tr[T_tr == 0]
        )

        mu0_hat[val_idx] = mu0_model.predict_proba(W_val)[:, 1]

        print(
            f"Fold {fold+1}: "
            f"mean PS={e_hat[val_idx].mean():.3f}"
        )

    # evitar divisiones explosivas
    e_hat = np.clip(e_hat, 0.01, 0.99)

    # ------------------------------------------------------
    # DR-AIPW
    # ------------------------------------------------------
    pseudo_outcome = (
        T * (Y - mu1_hat) / e_hat
        - (1 - T) * (Y - mu0_hat) / (1 - e_hat)
        + mu1_hat
        - mu0_hat
    )

    tau_hat = pseudo_outcome.mean()

    # ------------------------------------------------------
    # Bootstrap SE
    # ------------------------------------------------------
    rng = np.random.default_rng(seed)

    boot_taus = []

    for _ in range(n_boot):

        idx = rng.integers(0, n, size=n)

        boot_taus.append(
            pseudo_outcome[idx].mean()
        )

    se = np.std(boot_taus)

    return tau_hat, se

In [51]:


# ============================================================
# HELPER: OLS ATE
# ============================================================

def ols_ate(Y, T, X):

    X_reg = np.column_stack([T.reshape(-1,1), X])

    X_reg = sm.add_constant(X_reg)

    model = sm.OLS(Y, X_reg).fit()

    tau = model.params[1]
    se  = model.bse[1]

    return tau, se


In [52]:
# Oracle — para calcular tau_true por Monte Carlo (ya está en tau_true.npy)
# Solo verificacion
df_oracle = pd.read_csv("data/oracle_population.csv")
print(f"Oracle: {len(df_oracle):,} obs, bad rate: {df_oracle['y'].mean():.1%}")
print(f"Treatment rate en oracle: {df_oracle['T'].mean():.1%}")
print(f"ATE verdadero cargado   : {ATE_true:.4f}")

Oracle: 20,000 obs, bad rate: 44.8%
Treatment rate en oracle: 46.7%
ATE verdadero cargado   : 0.0178


In [53]:
# tau_true ya fue cargado en la celda 3 desde data/tau_true.npy
print(f"ATE verdadero (Monte Carlo): {ATE_true:.4f}")

ATE verdadero (Monte Carlo): 0.0178


## Benchmarks tables

Esa es la descomposición que te permite responder:

¿BASL ayuda?
(ii)−(i)
¿Control function ayuda?
(iii)−(i)
¿La combinación ayuda?
(iv)−(i)

y compararlas todas contra

In [54]:

# ============================================================
# COMPARISON OF METHODS
# ============================================================

print("=" * 60)
print("TRUE ATE")
print("=" * 60)
print(f"ATE_true = {ATE_true:.6f}")

# ------------------------------------------------------------
# (i) NAIVE
# Approved sample only
# No BASL
# No Heckman
# No DR-AIPW
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("(i) NAIVE")
print("=" * 60)

tau_naive, se_naive = ols_ate(
    Y_acc,
    T_acc,
    X_acc
)

print(f"tau = {tau_naive:+.6f}")
print(f"SE  = {se_naive:.6f}")
print(f"Bias= {tau_naive - ATE_true:+.6f}")

# ------------------------------------------------------------
# (ii) BASL ONLY
# Expanded sample
# No Heckman
# No DR-AIPW
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("(ii) BASL ONLY")
print("=" * 60)

tau_basl, se_basl = ols_ate(
    Y_exp,
    T_exp,
    X_exp
)

print(f"tau = {tau_basl:+.6f}")
print(f"SE  = {se_basl:.6f}")
print(f"Bias= {tau_basl - ATE_true:+.6f}")

# ------------------------------------------------------------
# (iii) CONTROL FUNCTION ONLY
# Approved sample
# Heckman + DR-AIPW
# No BASL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("(iii) CONTROL FUNCTION ONLY")
print("=" * 60)

lambda_acc_cf, _, _, _ = heckman_control_function(
    T_acc,
    X_acc,
    Z_acc
)

tau_cf, se_cf, _ = dr_aipw_crossfit(
    Y_acc,
    T_acc,
    X_acc,
    lambda_acc_cf
)

print(f"tau = {tau_cf:+.6f}")
print(f"SE  = {se_cf:.6f}")
print(f"Bias= {tau_cf - ATE_true:+.6f}")

# ------------------------------------------------------------
# (iv) FULL METHOD
# BASL + Heckman + DR-AIPW
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("(iv) FULL METHOD")
print("=" * 60)

tau_full, se_full, _ = dr_aipw_crossfit(
    Y_exp,
    T_exp,
    X_exp,
    lambda_exp
)

print(f"tau = {tau_full:+.6f}")
print(f"SE  = {se_full:.6f}")
print(f"Bias= {tau_full - ATE_true:+.6f}")

# ============================================================
# SUMMARY TABLE
# ============================================================

results = pd.DataFrame({
    "Method": [
        "Naive (OLS)",
        "BASL only (OLS)",
        "Control Function + DR",
        "Full Method"
    ],
    "ATE_hat": [
        tau_naive,
        tau_basl,
        tau_cf,
        tau_full
    ],
    "SE": [
        se_naive,
        se_basl,
        se_cf,
        se_full
    ],
    "Bias": [
        tau_naive - ATE_true,
        tau_basl - ATE_true,
        tau_cf - ATE_true,
        tau_full - ATE_true
    ]
})

results["AbsBias"] = np.abs(results["Bias"])

print("\n")
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print(results.round(6))

best = results.loc[
    results["AbsBias"].idxmin(),
    "Method"
]

print("\nBest estimator:")
print(best)

TRUE ATE
ATE_true = 0.017824

(i) NAIVE
tau = +0.061687
SE  = 0.008140
Bias= +0.043864

(ii) BASL ONLY
tau = +0.177772
SE  = 0.009229
Bias= +0.159948

(iii) CONTROL FUNCTION ONLY


First stage AUC: 0.5903
lambda mean: 0.9343
lambda std:  0.1371
  Fold 1: PS AUC=0.575
  Fold 2: PS AUC=0.560
  Fold 3: PS AUC=0.566
  Fold 4: PS AUC=0.591
  Fold 5: PS AUC=0.579
tau = +0.050306
SE  = 0.008624
Bias= +0.032482

(iv) FULL METHOD
  Fold 1: PS AUC=0.600
  Fold 2: PS AUC=0.588
  Fold 3: PS AUC=0.616
  Fold 4: PS AUC=0.625
  Fold 5: PS AUC=0.610
tau = +0.132133
SE  = 0.009084
Bias= +0.114309


FINAL COMPARISON
                  Method   ATE_hat        SE      Bias   AbsBias
0            Naive (OLS)  0.061687  0.008140  0.043864  0.043864
1        BASL only (OLS)  0.177772  0.009229  0.159948  0.159948
2  Control Function + DR  0.050306  0.008624  0.032482  0.032482
3            Full Method  0.132133  0.009084  0.114309  0.114309

Best estimator:
Control Function + DR
